<a href="https://colab.research.google.com/github/Yovany149/Conexiones/blob/main/Copia_de_Proyecto_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Proyecto ETL - Grupo Quetzal**

---



Grupo Quetzal es un conglomerado guatemalteco (ficticio) con tres unidades de negocio que crecieron por separado:

| Unidad de negocio | ¿A qué se dedica? |
|---|---|
| Banco Quetzal | Banca de personas: cuentas, depósitos a plazo, préstamos y campañas de colocación. |
| Quetzal Móvil | Operador de telefonía e internet: planes, contratos, cargos mensuales y cancelaciones (churn). |
| Tienda Quetzal | Cadena de retail con canal en línea: catálogo, facturas, líneas de detalle y devoluciones. |

Los tres conjuntos de datos son reales pero independientes: originalmente no comparten clientes entre sí. Para el caso
se generó un identificador maestro (customer_id) que vincula a una misma persona a través de las tres unidades; un
subconjunto de clientes tiene presencia en dos o en las tres.

# **Recursos necesarios**

Descargar los archivos a utilizar para el desarrollo del proyecto: [Archivos Proyecto ETL](https://github.com/VictorCuches/Curso_Procesamiento_Ingenieria_Python_2S2026/tree/master/ETL)


Repositorio curso: [Repositorio](https://github.com/VictorCuches/Curso_Procesamiento_Ingenieria_Python_2S2026.git)



# EXTRACT

## Importar archivos CSV

In [ ]:
import pandas as pd
import glob
import json
import os


In [ ]:
carpeta = "/content" # carpeta default Google Colab

archivos = glob.glob(os.path.join(carpeta, "*csv"))

dataframes_csv = {
    os.path.splitext(os.path.basename(archivo))[0]: pd.read_csv(archivo)
    for archivo in archivos
}

print(dataframes_csv.keys())

dict_keys(['stg_retail', 'stg_banca'])


In [ ]:
# DataFrame df_stg_banca
df_stg_banca = dataframes_csv['stg_banca']
df_stg_banca.head()

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y,customer_id
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no,CLI0000001
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no,CLI0000002
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no,CLI0000003
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no,CLI0000004
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no,CLI0000005


In [ ]:
# DataFrame df_stg_retail
df_stg_retail = dataframes_csv['stg_retail']
df_stg_retail.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,customer_id,es_cancelacion
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom,CLI0011888,0
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom,CLI0011888,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom,CLI0011888,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850,United Kingdom,CLI0011888,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850,United Kingdom,CLI0011888,0


## Importar archivo JSON

In [ ]:
# ❌❌❌ NO EJECUTAR!! SOLO ES EJEMPLO!! ❌❌❌
# Al realizar este tipo de importacion nos encontramos con el problema de acceso al json
# df_stg_cliente_maestro = pd.read_json('stg_cliente_maestro.json')
# df_stg_cliente_maestro.head()

In [ ]:
# Forma correcta para acceder y normalizar el json
with open("stg_cliente_maestro.json", "r", encoding="utf-8") as archivo:
  datos = json.load(archivo)

df_stg_cliente_maestro = pd.json_normalize(
    datos["stg_cliente_maestro"]
)

df_stg_cliente_maestro.head()

,customer_id,es_cliente_banco,es_cliente_telco,es_cliente_retail
0,CLI0000001,1,0,0
1,CLI0000002,1,0,0
2,CLI0000003,1,0,0
3,CLI0000004,1,0,0
4,CLI0000005,1,1,1


## Importar archivo TXT

In [ ]:
df_stg_telco = pd.read_csv("stg_telco.txt", sep="|")
df_stg_telco.head()

,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,customer_id,Unnamed: 23
0,NaN,----------,------,-------------,-------,----------,------,------------,----------------,---------------,...,-------------------,-------------------,--------------,----------------,-------------------------,--------------,------------,-----,-----------,NaN
1,NaN,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,CLI0031853,NaN
2,NaN,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,...,No,No,One year,No,Mailed check,56.95,1889.5,No,CLI0020865,NaN
3,NaN,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,CLI0001575,NaN
4,NaN,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,...,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No,CLI0026538,NaN


# TRANSFORM

## **Limpieza de df_stg_banca**

In [ ]:
display(df_stg_banca.head(3))

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y,customer_id
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no,CLI0000001
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no,CLI0000002
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no,CLI0000003


### Antes

In [ ]:
# Revisando tipos de datos y valores no nulos
df_stg_banca.info()

# Cantidad de nulos por columna
print(df_stg_banca.isnull().sum())

# Cantidad de registros duplicados
print(df_stg_banca.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   age          45211 non-null  int64 
 1   job          44923 non-null  object
 2   marital      45211 non-null  object
 3   education    43354 non-null  object
 4   default      45211 non-null  object
 5   balance      45211 non-null  int64 
 6   housing      45211 non-null  object
 7   loan         45211 non-null  object
 8   contact      32191 non-null  object
 9   day_of_week  45211 non-null  int64 
 10  month        45211 non-null  object
 11  duration     45211 non-null  int64 
 12  campaign     45211 non-null  int64 
 13  pdays        45211 non-null  int64 
 14  previous     45211 non-null  int64 
 15  poutcome     8252 non-null   object
 16  y            45211 non-null  object
 17  customer_id  45211 non-null  object
dtypes: int64(7), object(11)
memory usage: 6.2+ MB
age                0

### Procesamiento

In [ ]:
# Realizando copia limpia del DataFrame
df_stg_banca_clean = df_stg_banca.copy()

In [ ]:
# Tratamiento de valores nulos en diferentes columnas
columnas_con_nulos = ['job', 'education', 'contact', 'poutcome']

for columna in columnas_con_nulos:
    df_stg_banca_clean[columna] = df_stg_banca_clean[columna].fillna("unknown")

In [ ]:
print(df_stg_banca_clean.isnull().sum())
display(df_stg_banca_clean.head(3))

age            0
job            0
marital        0
education      0
default        0
balance        0
housing        0
loan           0
contact        0
day_of_week    0
month          0
duration       0
campaign       0
pdays          0
previous       0
poutcome       0
y              0
customer_id    0
dtype: int64


,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y,customer_id
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no,CLI0000001
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no,CLI0000002
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no,CLI0000003


In [ ]:
# Optimizando memoria
# Conversion de columnas de texto a tipo 'category'
columnas_texto = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'y']
for col in columnas_texto:
    df_stg_banca_clean[col] = df_stg_banca_clean[col].astype('category')

print(df_stg_banca_clean.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   age          45211 non-null  int64   
 1   job          45211 non-null  category
 2   marital      45211 non-null  category
 3   education    45211 non-null  category
 4   default      45211 non-null  category
 5   balance      45211 non-null  int64   
 6   housing      45211 non-null  category
 7   loan         45211 non-null  category
 8   contact      45211 non-null  category
 9   day_of_week  45211 non-null  int64   
 10  month        45211 non-null  category
 11  duration     45211 non-null  int64   
 12  campaign     45211 non-null  int64   
 13  pdays        45211 non-null  int64   
 14  previous     45211 non-null  int64   
 15  poutcome     45211 non-null  category
 16  y            45211 non-null  category
 17  customer_id  45211 non-null  object  
dtypes: category(10), int64(7),

In [ ]:
# Convertit columna 'balance' a float
df_stg_banca_clean['balance'] = df_stg_banca_clean['balance'].astype(float)
print(df_stg_banca_clean.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   age          45211 non-null  int64   
 1   job          45211 non-null  category
 2   marital      45211 non-null  category
 3   education    45211 non-null  category
 4   default      45211 non-null  category
 5   balance      45211 non-null  float64 
 6   housing      45211 non-null  category
 7   loan         45211 non-null  category
 8   contact      45211 non-null  category
 9   day_of_week  45211 non-null  int64   
 10  month        45211 non-null  category
 11  duration     45211 non-null  int64   
 12  campaign     45211 non-null  int64   
 13  pdays        45211 non-null  int64   
 14  previous     45211 non-null  int64   
 15  poutcome     45211 non-null  category
 16  y            45211 non-null  category
 17  customer_id  45211 non-null  object  
dtypes: category(10), float64(1

### Despues

In [ ]:
# Reemplazar el dataframe original con el limpio
df_stg_banca = df_stg_banca_clean

# Revisando tipos de datos y valores no nulos
df_stg_banca.info()

# Cantidad de nulos por columna
print(df_stg_banca.isnull().sum())

# Cantidad de registros duplicados
print(df_stg_banca.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   age          45211 non-null  int64   
 1   job          45211 non-null  category
 2   marital      45211 non-null  category
 3   education    45211 non-null  category
 4   default      45211 non-null  category
 5   balance      45211 non-null  float64 
 6   housing      45211 non-null  category
 7   loan         45211 non-null  category
 8   contact      45211 non-null  category
 9   day_of_week  45211 non-null  int64   
 10  month        45211 non-null  category
 11  duration     45211 non-null  int64   
 12  campaign     45211 non-null  int64   
 13  pdays        45211 non-null  int64   
 14  previous     45211 non-null  int64   
 15  poutcome     45211 non-null  category
 16  y            45211 non-null  category
 17  customer_id  45211 non-null  object  
dtypes: category(10), float64(1

## **Limpieza de df_stg_retail**

In [ ]:
display(df_stg_retail.head(3))

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,customer_id,es_cancelacion
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850,United Kingdom,CLI0011888,0
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850,United Kingdom,CLI0011888,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850,United Kingdom,CLI0011888,0


### Antes

In [ ]:
# Analisis inicial del DataFrame
df_stg_retail.info()

# Validacion de filas duplicadas
print("Filas duplicadas ", df_stg_retail.duplicated().sum())

# Cantidad de nulos por columna
print(df_stg_retail.isnull().sum())


# invoice_no, stock_code

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 406829 entries, 0 to 406828
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   InvoiceNo       406829 non-null  object 
 1   StockCode       406829 non-null  object 
 2   Description     406829 non-null  object 
 3   Quantity        406829 non-null  int64  
 4   InvoiceDate     406829 non-null  object 
 5   UnitPrice       406829 non-null  float64
 6   CustomerID      406829 non-null  int64  
 7   Country         406829 non-null  object 
 8   customer_id     406829 non-null  object 
 9   es_cancelacion  406829 non-null  int64  
dtypes: float64(1), int64(3), object(6)
memory usage: 31.0+ MB
Filas duplicadas  5225
InvoiceNo         0
StockCode         0
Description       0
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Country           0
customer_id       0
es_cancelacion    0
dtype: int64


In [ ]:
# copia temporal del DataFrame
df_stg_retail_clean = df_stg_retail.copy()

# limpiar duplicados
df_stg_retail_clean = df_stg_retail_clean.drop_duplicates()

# corregir el tipo de dato InvoiceDate a DateTime
df_stg_retail_clean['InvoiceDate'] = pd.to_datetime(df_stg_retail_clean['InvoiceDate'])

# optimizacion al tipo de dato Category
columnas_categoricas_retail = ['Country', 'es_cancelacion']
for col in columnas_categoricas_retail:
    df_stg_retail_clean[col] = df_stg_retail_clean[col].astype('category')

# Reemplazar el DataFrame al original\
df_stg_retail = df_stg_retail_clean


### Despues

In [ ]:
# Mostrando los resultados de la limpieza de df_stg_retail
df_stg_retail.info()
print("Filas duplicadas ", df_stg_retail.duplicated().sum())
print(df_stg_retail.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 401604 entries, 0 to 406828
Data columns (total 10 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   InvoiceNo       401604 non-null  object        
 1   StockCode       401604 non-null  object        
 2   Description     401604 non-null  object        
 3   Quantity        401604 non-null  int64         
 4   InvoiceDate     401604 non-null  datetime64[ns]
 5   UnitPrice       401604 non-null  float64       
 6   CustomerID      401604 non-null  int64         
 7   Country         401604 non-null  category      
 8   customer_id     401604 non-null  object        
 9   es_cancelacion  401604 non-null  category      
dtypes: category(2), datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 28.3+ MB
Filas duplicadas  0
InvoiceNo         0
StockCode         0
Description       0
Quantity          0
InvoiceDate       0
UnitPrice         0
CustomerID        0
Cou

## **Limpieza de df_stg_telco**

In [ ]:
# Explorando el DataFrame
display(df_stg_telco.head(3))

,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,customer_id,Unnamed: 23
0,NaN,----------,------,-------------,-------,----------,------,------------,----------------,---------------,...,-------------------,-------------------,--------------,----------------,-------------------------,--------------,------------,-----,-----------,NaN
1,NaN,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,...,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,CLI0031853,NaN
2,NaN,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,...,No,No,One year,No,Mailed check,56.95,1889.5,No,CLI0020865,NaN


### Antes

In [ ]:
df_stg_telco.info()
print("Filas duplicadas ", df_stg_telco.duplicated().sum())
print(df_stg_telco.isnull().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7044 entries, 0 to 7043
Data columns (total 24 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Unnamed: 0                 0 non-null      float64
 1   customerID                 7044 non-null   object 
 2   gender                     7044 non-null   object 
 3   SeniorCitizen              7044 non-null   object 
 4   Partner                    7044 non-null   object 
 5   Dependents                 7044 non-null   object 
 6   tenure                     7044 non-null   object 
 7   PhoneService               7044 non-null   object 
 8   MultipleLines              7044 non-null   object 
 9   InternetService            7044 non-null   object 
 10  OnlineSecurity             7044 non-null   object 
 11  OnlineBackup               7044 non-null   object 
 12  DeviceProtection           7044 non-null   object 
 13  TechSupport                7044 non-null   objec

### Procesamiento

In [ ]:
# creando una copia temporal para el DataFrame limpio
df_stg_telco_clean = df_stg_telco.copy()

# Eliminar la primera fila que tiene caracteres innecesarios
df_stg_telco_clean = df_stg_telco_clean.drop(0).reset_index(drop=True)

# Eliminar las columnas completamente nulas
df_stg_telco_clean = df_stg_telco_clean.drop(columns=['Unnamed: 0', 'Unnamed: 23'], errors='ignore')


display(df_stg_telco_clean.tail())


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,customer_id
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,One year,Yes,Mailed check,84.8,1990.5,No,CLI0042693
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,One year,Yes,Credit card (automatic),103.2,7362.9,No,CLI0011396
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.6,346.45,No,CLI0021855
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Mailed check,74.4,306.6,Yes,CLI0017177
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No,CLI0012541


In [ ]:
# verificando si al final del DataFrame hay filas innecesarias
display(df_stg_telco_clean.tail())

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,customer_id
7038,6840-RESVB,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,One year,Yes,Mailed check,84.8,1990.5,No,CLI0042693
7039,2234-XADUH,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,One year,Yes,Credit card (automatic),103.2,7362.9,No,CLI0011396
7040,4801-JZAZL,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.6,346.45,No,CLI0021855
7041,8361-LTMKD,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Mailed check,74.4,306.6,Yes,CLI0017177
7042,3186-AJIEK,Male,0,No,No,66,Yes,No,Fiber optic,Yes,...,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.65,6844.5,No,CLI0012541


In [ ]:
# Convertir columnas numericas a sus respectivos tipos
df_stg_telco_clean['tenure'] = pd.to_numeric(df_stg_telco_clean['tenure'], errors='coerce').fillna(0).astype(int)
df_stg_telco_clean['MonthlyCharges'] = pd.to_numeric(df_stg_telco_clean['MonthlyCharges'], errors='coerce').fillna(0.0)
df_stg_telco_clean['TotalCharges'] = pd.to_numeric(df_stg_telco_clean['TotalCharges'], errors='coerce').fillna(0.0)

In [ ]:
# Limpiar los espacios en blanco en los nombres de las columnas
df_stg_telco_clean.columns = df_stg_telco_clean.columns.str.strip()
df_stg_telco_clean.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'customer_id'],
      dtype='object')

In [ ]:
# Cambiar a tipo Category las respectivas columnas
columnas_cat_telco = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn'
]

for col in columnas_cat_telco:
    df_stg_telco_clean[col] = df_stg_telco_clean[col].astype('category')

# Guardar los datos limpios en DataFrame original
df_stg_telco = df_stg_telco_clean

### Despues

In [ ]:

df_stg_telco.info()
print("Filas duplicadas ", df_stg_telco.duplicated().sum())
print(df_stg_telco.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   customerID        7043 non-null   object  
 1   gender            7043 non-null   category
 2   SeniorCitizen     7043 non-null   category
 3   Partner           7043 non-null   category
 4   Dependents        7043 non-null   category
 5   tenure            7043 non-null   int64   
 6   PhoneService      7043 non-null   category
 7   MultipleLines     7043 non-null   category
 8   InternetService   7043 non-null   category
 9   OnlineSecurity    7043 non-null   category
 10  OnlineBackup      7043 non-null   category
 11  DeviceProtection  7043 non-null   category
 12  TechSupport       7043 non-null   category
 13  StreamingTV       7043 non-null   category
 14  StreamingMovies   7043 non-null   category
 15  Contract          7043 non-null   category
 16  PaperlessBilling  7043 n

## **Limpieza de df_stg_cliente_maestro**

In [ ]:
display(df_stg_cliente_maestro.head(3))

,customer_id,es_cliente_banco,es_cliente_telco,es_cliente_retail
0,CLI0000001,1,0,0
1,CLI0000002,1,0,0
2,CLI0000003,1,0,0


In [ ]:
df_stg_banca[df_stg_banca['customer_id'] == 'CLI0000001']

,age,job,marital,education,default,balance,housing,loan,contact,day_of_week,month,duration,campaign,pdays,previous,poutcome,y,customer_id
0,58,management,married,tertiary,no,2143.0,yes,no,unknown,5,may,261,1,-1,0,unknown,no,CLI0000001


### Antes

In [ ]:
print("Dimensiones: ", df_stg_cliente_maestro.shape)
print(df_stg_cliente_maestro.info())
print("Filas duplicadas ", df_stg_cliente_maestro.duplicated().sum())
print("Valores nulos por columna:")
print(df_stg_cliente_maestro.isnull().sum())

Dimensiones:  (45211, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   customer_id        45211 non-null  object
 1   es_cliente_banco   45211 non-null  int64 
 2   es_cliente_telco   45211 non-null  int64 
 3   es_cliente_retail  45211 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 1.4+ MB
None
Filas duplicadas  0
Valores nulos por columna:
customer_id          0
es_cliente_banco     0
es_cliente_telco     0
es_cliente_retail    0
dtype: int64


### Procesamiento

In [ ]:
# copia temporal para limpieza
df_stg_cliente_maestro_clean = df_stg_cliente_maestro.copy()

In [ ]:
# Eliminar duplicados si existieran
df_stg_cliente_maestro_clean = df_stg_cliente_maestro_clean.drop_duplicates()

# Cambio de tipo de columnas por Category
columnas_indicadores = ['es_cliente_banco', 'es_cliente_retail', 'es_cliente_telco']
for col in columnas_indicadores:
    df_stg_cliente_maestro_clean[col] = df_stg_cliente_maestro_clean[col].astype('category')

# Reemplazar el DataFrame original
df_stg_cliente_maestro = df_stg_cliente_maestro_clean

### Despues

In [ ]:
# Validando que la limpieza fue correcta
print("Dimensiones: ", df_stg_cliente_maestro.shape)
print(df_stg_cliente_maestro.info())
print("Filas duplicadas ", df_stg_cliente_maestro.duplicated().sum())

Dimensiones:  (45211, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   customer_id        45211 non-null  object  
 1   es_cliente_banco   45211 non-null  category
 2   es_cliente_telco   45211 non-null  category
 3   es_cliente_retail  45211 non-null  category
dtypes: category(3), object(1)
memory usage: 486.1+ KB
None
Filas duplicadas  0


## **Limpieza extras**

### Estandarización nombres de columnas


Vamos aplicar una normalizacion para que las columnas de los DataFrames sigan el formato estandar `snake_case` (minusculas y separados por guion)

Columnas de df_stg_telco

In [ ]:
df_stg_telco.columns


Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'customer_id'],
      dtype='object')

In [ ]:
import re

def clean_column_name(name):
  # Reemplazar cambios de minuscula a mayuscula por guion bajo
  name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name) # 'InvoiceNo' -> 'Invoice_No'

  # Reemplazar cualquier caracter no alfanumerico por guion bajo
  name = re.sub(r'[^a-zA-Z0-9]+', '_', name) # 'Invoice No' -> 'Invoice_No'

  # Limpiar guiones bajos extras y pasar a minusculas
  return name.strip('_').lower() # 'InvoiceNo' -> 'Invoice_No' -> 'invoice_no'

# Aplicar la normalizacion a df_stg_telco
df_stg_telco.columns = [clean_column_name(col) for col in df_stg_telco.columns]



In [ ]:
print("Columnas de df_stg_telco: ")
print(df_stg_telco.columns)

Columnas de df_stg_telco: 
Index(['customer_id', 'gender', 'senior_citizen', 'partner', 'dependents',
       'tenure', 'phone_service', 'multiple_lines', 'internet_service',
       'online_security', 'online_backup', 'device_protection', 'tech_support',
       'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
       'payment_method', 'monthly_charges', 'total_charges', 'churn',
       'customer_id'],
      dtype='object')


Columnas de df_stg_retail

In [ ]:
df_stg_retail.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'customer_id', 'es_cancelacion'],
      dtype='object')

In [ ]:
# Eliminar columna innecesaria
df_stg_retail = df_stg_retail.drop(columns=['customer_id'])

In [ ]:
# Aplicar la normalizacion a df_stg_retail
df_stg_retail.columns = [clean_column_name(col) for col in df_stg_retail.columns]

In [ ]:
df_stg_retail = df_stg_retail.rename(
    columns={
        'es_cancelacion': 'is_cancellation'
    }
)

In [ ]:
df_stg_retail.columns

Index(['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date',
       'unit_price', 'customer_id', 'country', 'customer_id',
       'is_cancellation'],
      dtype='object')

Columnas de df_stg_cliente_maestro

In [ ]:
df_stg_cliente_maestro.columns

Index(['customer_id', 'es_cliente_banco', 'es_cliente_telco',
       'es_cliente_retail'],
      dtype='object')

In [ ]:
df_stg_cliente_maestro = df_stg_cliente_maestro.rename(
    columns={
        'es_cliente_banco': 'is_customer_bank',
        'es_cliente_retail': 'is_customer_retail',
        'es_cliente_telco': 'is_customer_telco'
    }
)

In [ ]:
df_stg_cliente_maestro.columns

Index(['customer_id', 'is_customer_bank', 'is_customer_telco',
       'is_customer_retail'],
      dtype='object')

### Eliminar espacios en blanco

In [ ]:
# Identificando columnas que tiene espacios en blanco en sus registros
df_stg_telco.columns
df_stg_telco['gender'].unique()

['Female', 'Male  ']
Categories (2, object): ['Female', 'Male  ']

In [ ]:
# Quitando espacios en blanco de columna gender
df_stg_telco['gender'] = df_stg_telco['gender'].str.strip() # "Male   " -> "Male"
df_stg_telco['gender'].unique()

array(['Female', 'Male'], dtype=object)

In [ ]:
# Verificacion en todas las columnas de tipo Category en DataFrame
# // Usar este mismo bloque de codigo para validar despues de aplicar la limpieza
for columna in df_stg_telco.select_dtypes(include=['category']).columns:
    tiene_espacios = df_stg_telco[columna].dropna().astype(str).str.contains(r'^\s|\s$', regex=True).any()

    if tiene_espacios:
        print(f"La columna '{columna}' tiene valores con espacios al inicio o al final.")

La columna 'senior_citizen' tiene valores con espacios al inicio o al final.
La columna 'partner' tiene valores con espacios al inicio o al final.
La columna 'dependents' tiene valores con espacios al inicio o al final.
La columna 'phone_service' tiene valores con espacios al inicio o al final.
La columna 'multiple_lines' tiene valores con espacios al inicio o al final.
La columna 'internet_service' tiene valores con espacios al inicio o al final.
La columna 'online_security' tiene valores con espacios al inicio o al final.
La columna 'online_backup' tiene valores con espacios al inicio o al final.
La columna 'device_protection' tiene valores con espacios al inicio o al final.
La columna 'tech_support' tiene valores con espacios al inicio o al final.
La columna 'streaming_tv' tiene valores con espacios al inicio o al final.
La columna 'streaming_movies' tiene valores con espacios al inicio o al final.
La columna 'contract' tiene valores con espacios al inicio o al final.
La columna 'pa

In [ ]:
df_stg_telco['contract'].unique()

['Month-to-month', 'One year      ', 'Two year      ']
Categories (3, object): ['Month-to-month', 'One year      ', 'Two year      ']

In [ ]:
# Limpiando columnas con espacios en blanco en sus registros
columnas_texto = df_stg_telco.select_dtypes(include=['category']).columns

for columna in columnas_texto:
    df_stg_telco[columna] = df_stg_telco[columna].str.strip()

## LOAD

In [ ]:
# Conectar Google Drive con Colab
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
# Crear carpeta si no existe
import os
os.makedirs('/content/drive/MyDrive/Proyecto_ETL_Victor_Cuches/clean', exist_ok=True)

### Exportar CSV

In [ ]:
# Exportar csv
ruta = '/content/drive/MyDrive/Proyecto_ETL_Victor_Cuches/clean/df_stg_telco.csv'
df_stg_telco.to_csv(ruta, index=False, encoding='utf-8-sig')

In [ ]:
ruta = '/content/drive/MyDrive/Proyecto_ETL_Victor_Cuches/clean/df_stg_cliente_maestro.csv'
df_stg_cliente_maestro.to_csv(ruta, index=False, encoding='utf-8-sig')

### Exportar TXT

In [ ]:
# Exportar txt
ruta = '/content/drive/MyDrive/Proyecto_ETL_Victor_Cuches/clean/df_stg_banca.txt'
df_stg_banca.to_csv(
    ruta,
    index=False,
    sep='\t',
    encoding='utf-8'
)

### Exportar JSON

In [ ]:
ruta = '/content/drive/MyDrive/Proyecto_ETL_Victor_Cuches/clean/df_stg_retail.csv'
df_stg_retail.to_csv(ruta, index=False, encoding='utf-8-sig')